# AGORA — Data Processing Pipeline (`raw_train_set`, `raw_val_set`, `raw_test_set`)

Rigorous MELD extraction and audio standardization based on `ser_sft.ipynb` and `ser_grpo_v2.ipynb`:
- Loads official MELD splits (`train`, `dev` -> `val`, `test`).
- Standardizes all audio files in parallel to **16 kHz, mono (16-bit PCM WAV)** inside `DATA_DIR`.
- Applies **only basic preprocessing** (no laugh removal yet) to `raw_train_set`, `raw_val_set`, and `raw_test_set`.
- Saves standardized `raw_train_set.csv`, `raw_val_set.csv`, and `raw_test_set.csv` to `DATA_DIR`.


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from math import gcd
import os
import shutil
import subprocess
import time
from typing import Dict, Optional, Tuple

import numpy as np
import pandas as pd
import soundfile as sf
from tqdm.auto import tqdm


## 1. Paths & Configuration


In [ ]:
# Local MELD preprocessed directory (or Colab archive fallback)
MELD_DIR = Path("/home/fuganti/Projects/AFTA/data/prep_meld")
DRIVE_ARCHIVE = Path("/content/drive/MyDrive/SER_DPO/MELD/MELD.tar")  # Optional Colab fallback

# Output directory for standardized 16 kHz mono audio + split CSVs
DATA_DIR = Path("/home/fuganti/Projects/OPERA/data/meld_16k_mono")

TARGET_SR = 16000
TARGET_CHANNELS = 1
MAX_WORKERS = min(8, os.cpu_count() or 4)
OVERWRITE_AUDIO = False

# Mapping from canonical split name -> (MELD directory name, MELD CSV prefix)
SPLIT_CONFIG: Dict[str, Tuple[str, str]] = {
    "train": ("preprocessed_train_splits", "train"),
    "val": ("preprocessed_dev_splits_complete", "dev"),
    "test": ("preprocessed_output_repeated_splits_test", "test"),
}

DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f"MELD_DIR : {MELD_DIR} (exists={MELD_DIR.exists()})")
print(f"DATA_DIR : {DATA_DIR}")
print(f"Workers  : {MAX_WORKERS}")


## 2. MELD Extraction & Metadata Helpers
Consistent with `load_meld_split` / `load_meld` in `ser_sft.ipynb` and `ser_grpo_v2.ipynb`.


In [ ]:
def extract_tar(archive_path: Path, destination: Path) -> None:
    """Extract a .tar archive to destination."""
    archive_path = Path(archive_path)
    destination = Path(destination)

    if not archive_path.exists():
        raise FileNotFoundError(f"Archive not found: {archive_path}")

    destination.mkdir(parents=True, exist_ok=True)
    print(f"Extracting {archive_path.name} -> {destination}")
    start = time.time()

    subprocess.run(
        ["tar", "-xf", str(archive_path), "-C", str(destination)],
        check=True,
    )

    elapsed = time.time() - start
    print(f"Extraction completed in {elapsed:.1f}s")


def dataset_info(path: Path) -> None:
    """Print basic information about an extracted dataset."""
    path = Path(path)
    n_files = sum(1 for p in path.rglob("*") if p.is_file())
    size = subprocess.run(
        ["du", "-sh", str(path)],
        capture_output=True,
        text=True,
        check=True,
    ).stdout.split()[0]
    print(f"Directory: {path} | Files: {n_files:,} | Size: {size}")


def find_file(root: Path, pattern: str) -> Path:
    """Find the first file matching a pattern."""
    matches = list(Path(root).rglob(pattern))
    if not matches:
        raise FileNotFoundError(f"Could not find '{pattern}' under {root}")
    return matches[0]


def resolve_split_dir(destination: Path, split: str) -> Tuple[Path, str]:
    """Resolve split directory and MELD CSV prefix for train/val(dev)/test."""
    destination = Path(destination)
    if split in SPLIT_CONFIG:
        folder_name, csv_prefix = SPLIT_CONFIG[split]
        candidate = destination / folder_name
        if candidate.exists():
            return candidate, csv_prefix

    # Fallback patterns (e.g., preprocessed_dev_splits, dev_splits_complete, etc.)
    prefix = "dev" if split in ("val", "dev") else split
    candidates = [
        destination / f"preprocessed_{prefix}_splits",
        destination / f"preprocessed_{prefix}_splits_complete",
        destination / f"preprocessed_output_repeated_splits_{prefix}",
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate, prefix

    raise FileNotFoundError(f"Split directory for '{split}' not found under {destination}")


In [ ]:
def load_meld_split(split_dir: Path, split: str = "train") -> pd.DataFrame:
    """
    Load a MELD split and resolve audio paths.

    Expected files in split_dir:
        {split}_preprocessed.csv
        {split}_sent_emo.csv
        *.wav
    """
    split_dir = Path(split_dir)
    audio_files = list(split_dir.rglob("*.wav"))
    preprocessed_csv = find_file(split_dir, f"{split}_preprocessed.csv")
    emotion_csv = find_file(split_dir, f"{split}_sent_emo.csv")

    df = pd.read_csv(preprocessed_csv)
    df_emo = pd.read_csv(emotion_csv)

    # Resolve audio filenames -> absolute paths
    audio_map = {path.name: str(path.resolve()) for path in audio_files}
    df["audio_path"] = (
        df["audio_path"]
        .astype(str)
        .map(lambda x: Path(x).name)
        .map(audio_map)
    )

    # Normalize merge keys
    df["dialogue"] = df["dialogue"].astype(int)
    df["utterance"] = df["utterance"].astype(int)
    df_emo["Dialogue_ID"] = df_emo["Dialogue_ID"].astype(int)
    df_emo["Utterance_ID"] = df_emo["Utterance_ID"].astype(int)

    # Merge MELD metadata
    df = df.merge(
        df_emo[["Dialogue_ID", "Utterance_ID", "Emotion", "Utterance", "Speaker"]],
        left_on=["dialogue", "utterance"],
        right_on=["Dialogue_ID", "Utterance_ID"],
        how="left",
    )

    df = (
        df.drop(columns=["Dialogue_ID", "Utterance_ID"])
        .rename(
            columns={
                "Utterance": "text",
                "Emotion": "emotion",
                "Speaker": "speaker",
            }
        )
    )

    if "time" in df.columns and "duration" not in df.columns:
        df["duration"] = pd.to_numeric(df["time"], errors="coerce")

    missing_audio = int(df["audio_path"].isna().sum())
    missing_emotion = int(df["emotion"].isna().sum())

    print(f"[{split}] Loaded {len(df):,} samples")
    print(f"[{split}] Audio matched   : {len(df) - missing_audio:,}/{len(df):,}")
    if missing_audio:
        print(f"[{split}] Warning: {missing_audio:,} audio files missing")
    print(f"[{split}] Metadata matched: {len(df) - missing_emotion:,}/{len(df):,}")
    if missing_emotion:
        print(f"[{split}] Warning: {missing_emotion:,} metadata rows missing")

    return df


def load_meld(
    archive_path: Path = DRIVE_ARCHIVE,
    destination: Path = MELD_DIR,
    split: str = "train",
) -> pd.DataFrame:
    """Load (and optionally extract) a MELD split."""
    destination = Path(destination)
    if not destination.exists():
        extract_tar(archive_path, destination)

    split_dir, csv_prefix = resolve_split_dir(destination, split)
    return load_meld_split(split_dir, csv_prefix)


## 3. Load Raw Splits (`raw_train_set`, `raw_val_set`, `raw_test_set`)


In [ ]:
dataset_info(MELD_DIR)

raw_train_set = load_meld(archive_path=DRIVE_ARCHIVE, destination=MELD_DIR, split="train")
raw_val_set = load_meld(archive_path=DRIVE_ARCHIVE, destination=MELD_DIR, split="val")
raw_test_set = load_meld(archive_path=DRIVE_ARCHIVE, destination=MELD_DIR, split="test")


## 3.1 Speaker Gender Classification (`F` / `M`) & Training Set Filtering
Classifies speakers into Female (`"F"`) and Male (`"M"`) using `KNOWN_SPEAKERS_GENDER` (Layer 1) and explicit gender keywords (`FEMALE_KEYWORDS`, `MALE_KEYWORDS`, Layer 2). Utterances whose speaker is not resolved by these rules (`gender is None`) are removed from the training set (`raw_train_set` / `clean_train_set`).


In [ ]:
KNOWN_SPEAKERS_GENDER = {
    # Protagonistas (~83% do MELD)
    "Rachel": "F", "Monica": "F", "Phoebe": "F",
    "Joey": "M", "Chandler": "M", "Ross": "M",
    
    # Secundários recorrentes (Femininos)
    "Janice": "F", "Carol": "F", "Susan": "F", "Judy": "F", "Emily": "F",
    "Ursula": "F", "Mona": "F", "Charlie": "F", "Amy": "F", "Jill": "F",
    "Mindy": "F", "Estelle": "F", "Kathy": "F", "Elizabeth": "F", "Nora": "F",
    "Julie": "F", "Alice": "F", "Joanna": "F", "Lydia": "F", "Caitlin": "F",
    "Bonnie": "F", "Erin": "F", "Chloe": "F", "Dina": "F", "Kim": "F",
    "Leslie": "F", "ginger": "F", "Ginger": "F", "Aurora": "F", "Kristen": "F",
    
    # Secundários recorrentes (Masculinos)
    "Gunther": "M", "Richard": "M", "Mike": "M", "Jack": "M", "Ben": "M",
    "Pete": "M", "David": "M", "Paul": "M", "Frank": "M", "Tag": "M",
    "Joshua": "M", "Gary": "M", "Paolo": "M", "Eddie": "M", "Mark": "M",
    "Barry": "M", "Treeger": "M", "Gavin": "M", "Danny": "M", "Will": "M",
    "Leonard": "M", "parker": "M", "Parker": "M", "Steve": "M", "stu": "M",
    "Stu": "M", "Doug": "M", "Terry": "M", "Allesandro": "M", "Wayne": "M",
}

# ==========================================
# CAMADA 2: Títulos e papéis com gênero explícito no nome
# ==========================================
FEMALE_KEYWORDS = [
    "mrs.", "ms.", "miss", "woman", "girl", "waitress", "hostess",
    "nurse", "mother", "mom", "grandmother", "sister", "lady",
    "stewardess", "actress", "bride", "her", "she"
]
MALE_KEYWORDS = [
    "mr.", "man", "guy", "boy", "waiter", "policeman", "fireman",
    "father", "dad", "grandfather", "brother", "gentleman", "actor",
    "groom", "bellhop", "doorman"
]

def get_lexical_gender(speaker_name: str):
    """Tenta resolver o gênero pelo dicionário de personagens ou palavras-chave."""
    spk_clean = str(speaker_name).strip()
    if spk_clean in KNOWN_SPEAKERS_GENDER:
        return KNOWN_SPEAKERS_GENDER[spk_clean]
    
    spk_lower = spk_clean.lower()
    for kw in FEMALE_KEYWORDS:
        if kw in spk_lower.split() or spk_lower.startswith(kw):
            return "F"
    for kw in MALE_KEYWORDS:
        if kw in spk_lower.split() or spk_lower.startswith(kw):
            return "M"
    return None  # Vai para a Camada 3 (Acústica)


def classify_and_filter_gender(
    df: pd.DataFrame,
    split_name: str,
    drop_unknown: bool = False,
) -> pd.DataFrame:
    """Add 'gender' column ('F'/'M') using get_lexical_gender and optionally drop unresolved rows."""
    out = df.copy()
    out["gender"] = out["speaker"].map(get_lexical_gender)
    n_total = len(out)
    n_f = int((out["gender"] == "F").sum())
    n_m = int((out["gender"] == "M").sum())
    n_unk = int(out["gender"].isna().sum())

    if drop_unknown:
        out = out.loc[out["gender"].isin(["F", "M"])].reset_index(drop=True)
        print(
            f"[{split_name}] Gender classification: F={n_f:,}, M={n_m:,} | "
            f"Removed unresolved: {n_unk:,}/{n_total:,} -> Retained: {len(out):,} ({100.0 * len(out) / max(1, n_total):.2f}%)"
        )
    else:
        print(
            f"[{split_name}] Gender classification: F={n_f:,}, M={n_m:,}, Unresolved={n_unk:,}/{n_total:,} (kept for eval)"
        )
    return out


# Apply to raw splits (removing unresolved speakers from raw_train_set)
raw_train_set = classify_and_filter_gender(raw_train_set, "raw_train_set", drop_unknown=True)
raw_val_set = classify_and_filter_gender(raw_val_set, "raw_val_set", drop_unknown=False)
raw_test_set = classify_and_filter_gender(raw_test_set, "raw_test_set", drop_unknown=False)

# If clean_train_set was already created in session, filter it as well
if "clean_train_set" in globals() and isinstance(clean_train_set, pd.DataFrame):
    clean_train_set = classify_and_filter_gender(clean_train_set, "clean_train_set", drop_unknown=True)


## 4. Parallel Standardization to 16 kHz Mono WAV in `DATA_DIR`
All splits (`train`, `val`, `test`) receive the same basic treatment:
- Convert multi-channel audio to mono by channel averaging.
- Resample to `16000 Hz` if needed.
- Save as 16-bit PCM `.wav` under `DATA_DIR / split / dia{dialogue}_utt{utterance}.wav`.
- Update `audio_path` and `duration` in `raw_train_set`, `raw_val_set`, and `raw_test_set`.
- No laugh removal or denoising is applied here.


In [ ]:
def _resample_audio(wav: np.ndarray, orig_sr: int, target_sr: int) -> np.ndarray:
    if orig_sr == target_sr:
        return wav
    try:
        import librosa
        return librosa.resample(wav, orig_sr=orig_sr, target_sr=target_sr).astype(np.float32)
    except ImportError:
        pass
    try:
        from scipy.signal import resample_poly
        divisor = gcd(orig_sr, target_sr)
        return resample_poly(wav, target_sr // divisor, orig_sr // divisor).astype(np.float32)
    except ImportError:
        n_out = int(round(len(wav) * target_sr / orig_sr))
        x_old = np.linspace(0.0, 1.0, num=len(wav), endpoint=False)
        x_new = np.linspace(0.0, 1.0, num=n_out, endpoint=False)
        return np.interp(x_new, x_old, wav).astype(np.float32)


def _process_single_audio(task: Tuple[int, str, Path]) -> Tuple[int, Optional[str], float, Optional[str]]:
    idx, src_path, dst_path = task
    if not src_path or not Path(src_path).exists():
        return idx, None, float("nan"), f"Missing source: {src_path}"

    try:
        if dst_path.exists() and not OVERWRITE_AUDIO:
            info = sf.info(str(dst_path))
            if info.samplerate == TARGET_SR and info.channels == TARGET_CHANNELS:
                return idx, str(dst_path.resolve()), float(info.duration), None

        dst_path.parent.mkdir(parents=True, exist_ok=True)
        src_info = sf.info(str(src_path))

        # Fast path: already 16 kHz, mono, PCM_16 WAV
        if (
            src_info.samplerate == TARGET_SR
            and src_info.channels == TARGET_CHANNELS
            and src_info.subtype == "PCM_16"
            and Path(src_path).suffix.lower() == ".wav"
        ):
            shutil.copy2(src_path, dst_path)
            return idx, str(dst_path.resolve()), float(src_info.duration), None

        wav, sr = sf.read(str(src_path), dtype="float32", always_2d=True)
        if wav.shape[1] > 1:
            wav = wav.mean(axis=1)
        else:
            wav = wav[:, 0]

        if sr != TARGET_SR:
            wav = _resample_audio(wav, sr, TARGET_SR)

        wav = np.clip(wav, -1.0, 1.0)
        sf.write(str(dst_path), wav, TARGET_SR, subtype="PCM_16")
        duration = float(len(wav) / TARGET_SR)
        return idx, str(dst_path.resolve()), duration, None
    except Exception as exc:
        return idx, None, float("nan"), str(exc)


def export_split_16k_mono(
    df: pd.DataFrame,
    split_name: str,
    data_dir: Path = DATA_DIR,
    max_workers: int = MAX_WORKERS,
) -> pd.DataFrame:
    """Standardize all audio in a split to 16 kHz mono WAV in parallel."""
    out_df = df.copy().reset_index(drop=True)
    split_audio_dir = Path(data_dir) / split_name
    split_audio_dir.mkdir(parents=True, exist_ok=True)

    tasks = []
    for idx, row in out_df.iterrows():
        src_path = row["audio_path"]
        fname = f"dia{int(row['dialogue'])}_utt{int(row['utterance'])}.wav"
        dst_path = split_audio_dir / fname
        tasks.append((idx, src_path, dst_path))

    new_paths = [None] * len(out_df)
    durations = [float("nan")] * len(out_df)
    errors = []

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(_process_single_audio, t) for t in tasks]
        for fut in tqdm(as_completed(futures), total=len(futures), desc=f"Exporting {split_name} (16k mono)"):
            idx, out_path, dur, err = fut.result()
            new_paths[idx] = out_path
            durations[idx] = dur
            if err is not None:
                errors.append((idx, err))

    out_df["source_audio_path"] = out_df["audio_path"]
    out_df["audio_path"] = new_paths
    out_df["duration"] = durations

    # Basic hygiene: drop rows whose audio failed or is missing
    valid_mask = out_df["audio_path"].notna() & out_df["emotion"].notna()
    dropped = int((~valid_mask).sum())
    if dropped or errors:
        print(f"[{split_name}] Dropped {dropped} invalid rows ({len(errors)} audio errors)")
        for idx, err in errors[:5]:
            print(f"  - row {idx}: {err}")
    out_df = out_df.loc[valid_mask].reset_index(drop=True)

    csv_path = Path(data_dir) / f"raw_{split_name}_set.csv"
    out_df.to_csv(csv_path, index=False)
    print(f"[{split_name}] Saved {len(out_df):,} rows -> {csv_path}")
    return out_df


In [ ]:
raw_train_set = export_split_16k_mono(raw_train_set, "train", DATA_DIR)
raw_val_set = export_split_16k_mono(raw_val_set, "val", DATA_DIR)
raw_test_set = export_split_16k_mono(raw_test_set, "test", DATA_DIR)


## 5. Verification & Split Summary


In [ ]:
def verify_split(df: pd.DataFrame, name: str, sample_check: int = 100) -> None:
    assert df["audio_path"].isna().sum() == 0, f"{name}: NaN audio_path found"
    assert df["emotion"].isna().sum() == 0, f"{name}: NaN emotion found"

    check_paths = df["audio_path"].head(sample_check).tolist()
    bad_sr, bad_ch = 0, 0
    for p in check_paths:
        info = sf.info(p)
        if info.samplerate != TARGET_SR:
            bad_sr += 1
        if info.channels != TARGET_CHANNELS:
            bad_ch += 1

    print(f"=== {name.upper()} ({len(df):,} samples) ===")
    print(f"Checked {len(check_paths)} files -> bad_sr={bad_sr}, bad_ch={bad_ch}")
    print(f"Duration (s): min={df['duration'].min():.2f}, median={df['duration'].median():.2f}, max={df['duration'].max():.2f}")
    print("Emotions:")
    print(df["emotion"].value_counts().to_string())
    print()


verify_split(raw_train_set, "raw_train_set")
verify_split(raw_val_set, "raw_val_set")
verify_split(raw_test_set, "raw_test_set")

# Check dialogue-level overlap across splits (official MELD splits should be disjoint)
train_dias = set(raw_train_set["dialogue"].unique())
val_dias = set(raw_val_set["dialogue"].unique())
test_dias = set(raw_test_set["dialogue"].unique())
print("Dialogue counts -> train:", len(train_dias), "val:", len(val_dias), "test:", len(test_dias))


## 6. EMO-TAG Single-Utterance Cleaning & Background Laughter Removal
Implements the exact 5-step single-utterance audio cleaning pipeline from **EMO-TAG** (*Finding Emotions Where They Belong: Rethinking Audio Emotion Recognition through Masked Temporal Affective Grounding*, Mohamed, Hansen, & Tan, 2026 — `Section 3.1`, `Appendix A.2`, and `Figure 4`):

1. **Input & Transcript Normalization (`Appendix A.2`):**
   - Process audio as `16 kHz` mono waveforms.
   - Lowercase reference transcripts, strip punctuation except apostrophes (`'`), and normalize whitespace.
2. **VAD Segmentation (`Section 3.1`, `Figure 4` Step 2):**
   - Segment each utterance into candidate **speech** and **non-speech** regions using **rVAD** (Tan et al., 2020) and reject empty/inaudible recordings.
3. **First-Pass WhisperX Alignment & Unrelated Speech Removal (`Section 3.1`, `Appendix A.2`, `Figure 4` Step 3):**
   - Transcribe each candidate speech region with **WhisperX** (`whisper-large-v3`) and run forced alignment with the ground-truth transcript.
   - Discard any speech segment that has **zero token overlap** with the ground-truth utterance (removing other speakers / overlapping dialogue).
4. **High-Energy Laughter Removal in Non-Speech Regions (`Section 3.1`, `Appendix A.2`, `Figure 4` Step 4):**
   - For non-speech intervals between selected speech regions, compute **RMS level (dBFS)**, **peak amplitude (dBFS)**, frame-level energy variation, and proportion of voiced frames.
   - **Retain** intervals as natural pauses or sighs **only when `RMS <= -35 dBFS` and `Peak <= -25 dBFS`**.
   - **Remove** high-energy non-speech regions (`RMS > -35 dBFS` or `Peak > -25 dBFS`, i.e., sitcom background laughter tracks).
   - If selected speech regions and retained pause/sigh intervals are contiguous, crop a single continuous span; otherwise concatenate the retained regions.
5. **Second-Pass WhisperX Forced Alignment, Boundary Trimming & Edit-Distance Gate (`Section 3.1`, `Appendix A.2`, `Figure 4` Step 5):**
   - Run **WhisperX** forced alignment again on the retained waveform and crop from the start of the first aligned word to the end of the last aligned word (with small side margins), accepting the refined crop only if it preserves the same transcription.
   - **Reject the sample** if the final aligned transcript has **$\ge 2$ consecutive insertions** (extra words not in the reference utterance) or **$\ge 3$ replacements** (substituted/misrecognized words), or if the resulting audio is empty/inaudible.


In [ ]:
# Optional dependency setup for WhisperX and rVAD (run if not already installed)
import importlib.util
import sys

RVAD_REPO_DIR = Path("/home/fuganti/Projects/OPERA/third_party/rVAD")

def ensure_emo_tag_dependencies(install_whisperx: bool = False, clone_rvad: bool = True) -> None:
    if clone_rvad and not RVAD_REPO_DIR.exists():
        try:
            RVAD_REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
            subprocess.run(
                ["git", "clone", "--depth", "1", "https://github.com/zhenghuatan/rVAD.git", str(RVAD_REPO_DIR)],
                check=True,
            )
            print(f"Cloned rVAD into {RVAD_REPO_DIR}")
        except Exception as exc:
            print(f"Note: rVAD git clone skipped ({exc}); using built-in rVAD-compatible segmenter.")

    if RVAD_REPO_DIR.exists() and str(RVAD_REPO_DIR) not in sys.path:
        sys.path.insert(0, str(RVAD_REPO_DIR))

    if install_whisperx and importlib.util.find_spec("whisperx") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "whisperx"], check=True)


In [ ]:
from dataclasses import dataclass, asdict
import difflib
import re
from typing import Any, List, Sequence

# EMO-TAG Appendix A.2 & Section 3.1 exact constants
EMO_TAG_SR = 16000
PAUSE_MAX_RMS_DBFS = -35.0          # Appendix A.2: retain pauses/sighs when RMS <= -35 dBFS
PAUSE_MAX_PEAK_DBFS = -25.0         # Appendix A.2: retain pauses/sighs when Peak <= -25 dBFS
BOUNDARY_MARGIN_SEC = 0.05          # Appendix A.2: small margins on both sides of first/last aligned word
MAX_CONSECUTIVE_INSERTIONS = 2      # Section 3.1: reject if >= 2 consecutive insertions
MAX_REPLACEMENTS = 3                # Section 3.1: reject if >= 3 replacements (substitutions)
MIN_SPEECH_DURATION_SEC = 0.15      # Minimum audible speech span duration

CLEAN_DATA_DIR = DATA_DIR / "emo_tag_clean"
CLEAN_DATA_DIR.mkdir(parents=True, exist_ok=True)


@dataclass
class IntervalMetrics:
    start_sec: float
    end_sec: float
    is_speech: bool
    rms_dbfs: float
    peak_dbfs: float
    frame_energy_var: float
    voiced_frame_ratio: float


def normalize_emo_tag_transcript(text: str) -> str:
    """Appendix A.2: lowercase, remove punctuation except apostrophes, and normalize whitespace."""
    if not isinstance(text, str):
        return ""
    # Normalize curly apostrophes to standard ASCII apostrophe
    text = text.replace("’", "'").replace("‘", "'").lower()
    # Remove all punctuation/symbols except word characters, whitespace, and apostrophes
    text = re.sub(r"[^a-z0-9'\s]+", " ", text)
    # Collapse repeated apostrophes or standalone apostrophes
    text = re.sub(r"(?<!\w)'|'(?!\w)", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def tokenize_emo_tag(text: str) -> List[str]:
    norm = normalize_emo_tag_transcript(text)
    return norm.split() if norm else []


def compute_interval_acoustic_metrics(
    wav: np.ndarray,
    sr: int = EMO_TAG_SR,
    start_sec: float = 0.0,
    end_sec: Optional[float] = None,
    is_speech: bool = False,
) -> IntervalMetrics:
    """Appendix A.2: compute RMS (dBFS), peak (dBFS), frame-level energy variation, and voiced frame ratio."""
    eps = 1e-10
    if end_sec is None:
        end_sec = len(wav) / float(sr)
    if len(wav) == 0:
        return IntervalMetrics(start_sec, end_sec, is_speech, -120.0, -120.0, 0.0, 0.0)

    rms = float(np.sqrt(np.mean(np.square(wav, dtype=np.float64))))
    peak = float(np.max(np.abs(wav)))
    rms_dbfs = float(20.0 * np.log10(max(rms, eps)))
    peak_dbfs = float(20.0 * np.log10(max(peak, eps)))

    # Frame-level analysis (25 ms window, 10 ms hop at 16 kHz)
    win_len = int(0.025 * sr)
    hop_len = int(0.010 * sr)
    if len(wav) < win_len:
        frames = wav[None, :]
    else:
        n_frames = 1 + (len(wav) - win_len) // hop_len
        frames = np.lib.stride_tricks.sliding_window_view(wav, win_len)[::hop_len][:n_frames]

    frame_rms_db = 20.0 * np.log10(np.sqrt(np.mean(np.square(frames, dtype=np.float64), axis=1)) + eps)
    frame_energy_var = float(np.var(frame_rms_db)) if len(frame_rms_db) > 1 else 0.0

    # Zero-crossing rate + autocorrelation voicing estimate per frame
    zcr = np.mean(np.abs(np.diff(np.signbit(frames), axis=1)), axis=1)
    voiced_mask = (frame_rms_db > -45.0) & (zcr < 0.25)
    voiced_frame_ratio = float(np.mean(voiced_mask)) if len(voiced_mask) > 0 else 0.0

    return IntervalMetrics(
        start_sec=start_sec,
        end_sec=end_sec,
        is_speech=is_speech,
        rms_dbfs=rms_dbfs,
        peak_dbfs=peak_dbfs,
        frame_energy_var=frame_energy_var,
        voiced_frame_ratio=voiced_frame_ratio,
    )


def is_natural_pause_or_sigh(
    metrics: IntervalMetrics,
    max_rms_dbfs: float = PAUSE_MAX_RMS_DBFS,
    max_peak_dbfs: float = PAUSE_MAX_PEAK_DBFS,
) -> bool:
    """Appendix A.2: retain intervening non-speech intervals as natural pauses/sighs iff RMS <= -35 dBFS and Peak <= -25 dBFS."""
    return (metrics.rms_dbfs <= max_rms_dbfs) and (metrics.peak_dbfs <= max_peak_dbfs)


def rvad_frame_decisions(wav: np.ndarray, sr: int = EMO_TAG_SR) -> np.ndarray:
    """
    Run rVAD (Tan et al., 2020) frame-level voice activity detection (10 ms frame shift).
    Uses official speechproc/rVADfast if present, otherwise uses an rVAD-equivalent
    two-pass spectral flatness + high-pass energy + hangover detector.
    """
    if len(wav) < int(0.025 * sr):
        return np.zeros(0, dtype=np.int32)

    # Try official rVAD implementation if cloned in RVAD_REPO_DIR
    try:
        import speechproc
        winlen, ovrlen = 0.025, 0.010
        pre_coef, nfilter, nftt = 0.97, 20, 512
        ftThres, vadThres = 0.5, 0.4
        nfr10 = int(np.floor((len(wav) - (winlen - ovrlen) * sr) / (ovrlen * sr)))
        if nfr10 > 0:
            ft, flen, fsh10, nfr10 = speechproc.sflux(wav, sr, winlen, ovrlen, nftt)
            pv01 = np.zeros(nfr10)
            pv01[np.less_equal(ft, ftThres)] = 1
            pitch = copy_pv = pv01.copy()
            pvblk = speechproc.pitchblockdetect(pv01, pitch, nfr10, 1)
            fdata = speechproc.highpass(wav, sr)
            noise_samp, noise_seg, n_noise_samp = speechproc.snre_highenergy(
                fdata, nfr10, flen, fsh10, np.round( flen * 0.5 ), pv01, pvblk
            )
            for j in range(n_noise_samp):
                fdata[range(int(noise_samp[j, 0]), int(noise_samp[j, 1]) + 1)] = 0
            vad_seg = speechproc.snre_vad(
                fdata, nfr10, flen, fsh10, np.round(flen * 0.5), pv01, pvblk, vadThres
            )
            return np.asarray(vad_seg, dtype=np.int32)
    except Exception:
        pass

    # Built-in rVAD-equivalent two-pass segment-based spectral flatness + SNR-energy detector
    win_len = int(0.025 * sr)
    hop_len = int(0.010 * sr)
    n_frames = 1 + (len(wav) - win_len) // hop_len
    if n_frames <= 0:
        return np.zeros(0, dtype=np.int32)

    # Pre-emphasis / high-pass filtering (removes DC & low-frequency rumble < 60 Hz)
    hp_wav = np.append(wav[0], wav[1:] - 0.97 * wav[:-1])
    frames = np.lib.stride_tricks.sliding_window_view(hp_wav, win_len)[::hop_len][:n_frames]
    window = np.hamming(win_len).astype(np.float32)
    spec = np.abs(np.fft.rfft(frames * window[None, :], n=512, axis=1)) + 1e-10

    # Spectral flatness (geometric mean / arithmetic mean)
    geo_mean = np.exp(np.mean(np.log(spec), axis=1))
    arith_mean = np.mean(spec, axis=1)
    flatness = geo_mean / arith_mean

    # Short-time frame energy (dBFS)
    raw_frames = np.lib.stride_tricks.sliding_window_view(wav, win_len)[::hop_len][:n_frames]
    rms_db = 20.0 * np.log10(np.sqrt(np.mean(np.square(raw_frames, dtype=np.float64), axis=1)) + 1e-10)

    # Pass 1: estimate noise floor from high-flatness / low-energy frames
    noise_floor_db = float(np.percentile(rms_db, 15))
    snr_db = rms_db - max(noise_floor_db, -65.0)
    speech_frames = (rms_db > -45.0) & ((snr_db > 6.0) | ((flatness < 0.45) & (rms_db > -38.0)))

    # Morphological hangover smoothing (bridge < 150 ms gaps, drop < 80 ms spikes)
    decisions = speech_frames.astype(np.int32)
    # Bridge short gaps (<= 15 frames = 150 ms)
    idx_ones = np.where(decisions == 1)[0]
    if len(idx_ones) > 1:
        for a, b in zip(idx_ones[:-1], idx_ones[1:]):
            if 1 < (b - a) <= 15:
                decisions[a:b] = 1
    # Remove isolated micro-bursts (< 8 frames = 80 ms)
    i = 0
    while i < len(decisions):
        if decisions[i] == 1:
            j = i
            while j < len(decisions) and decisions[j] == 1:
                j += 1
            if (j - i) < 8:
                decisions[i:j] = 0
            i = j
        else:
            i += 1

    return decisions


def segment_audio_with_rvad(
    wav: np.ndarray,
    sr: int = EMO_TAG_SR,
) -> List[IntervalMetrics]:
    """Section 3.1 Step 2: Partition audio into speech and non-speech regions using rVAD."""
    total_dur = len(wav) / float(sr)
    decisions = rvad_frame_decisions(wav, sr=sr)
    if len(decisions) == 0 or not np.any(decisions == 1):
        return []

    hop_sec = 0.010
    intervals: List[IntervalMetrics] = []
    n = len(decisions)
    i = 0
    while i < n:
        state = bool(decisions[i] == 1)
        j = i
        while j < n and (decisions[j] == 1) == state:
            j += 1
        start_sec = round(i * hop_sec, 4)
        end_sec = round(min(total_dur, j * hop_sec if j < n else total_dur), 4)
        s_idx = int(round(start_sec * sr))
        e_idx = int(round(end_sec * sr))
        seg_wav = wav[s_idx:e_idx]
        if len(seg_wav) > 0:
            intervals.append(
                compute_interval_acoustic_metrics(
                    seg_wav, sr=sr, start_sec=start_sec, end_sec=end_sec, is_speech=state
                )
            )
        i = j

    return intervals


def compute_word_edit_diagnostics(
    hyp_tokens: Sequence[str],
    ref_tokens: Sequence[str],
) -> Dict[str, Any]:
    """
    Section 3.1 Step 5: Compare WhisperX transcript tokens against ground-truth reference tokens.
    Computes:
      - token_overlap_count: number of words matching reference
      - max_consecutive_insertions: max consecutive extra words not in reference (reject if >= 2)
      - total_replacements: total substituted/replaced words (reject if >= 3)
    """
    ref_set = set(ref_tokens)
    overlap_count = sum(1 for tok in hyp_tokens if tok in ref_set)

    matcher = difflib.SequenceMatcher(a=list(ref_tokens), b=list(hyp_tokens), autojunk=False)
    max_consec_ins = 0
    total_replacements = 0

    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag == "insert":
            ins_len = j2 - j1
            if ins_len > max_consec_ins:
                max_consec_ins = ins_len
        elif tag == "replace":
            ref_len = i2 - i1
            hyp_len = j2 - j1
            replaced = min(ref_len, hyp_len)
            total_replacements += replaced
            extra_ins = max(0, hyp_len - ref_len)
            if extra_ins > max_consec_ins:
                max_consec_ins = extra_ins

    reject_insertions = max_consec_ins >= MAX_CONSECUTIVE_INSERTIONS
    reject_replacements = total_replacements >= MAX_REPLACEMENTS

    return {
        "token_overlap_count": int(overlap_count),
        "max_consecutive_insertions": int(max_consec_ins),
        "total_replacements": int(total_replacements),
        "reject_insertions": bool(reject_insertions),
        "reject_replacements": bool(reject_replacements),
        "passes_edit_gate": bool(not reject_insertions and not reject_replacements and overlap_count > 0),
    }


In [ ]:
class EmoTagWhisperXCleaner:
    """
    Full EMO-TAG Single-Utterance Audio Cleaner (Mohamed et al., 2026, Section 3.1 & Appendix A.2).
    Combines rVAD segmentation, high-energy non-speech laughter removal (RMS <= -35 dBFS, Peak <= -25 dBFS),
    two-pass WhisperX (whisper-large-v3) transcription + forced alignment, boundary trimming,
    and word-level edit-distance rejection (>= 2 consecutive insertions or >= 3 replacements).
    """

    def __init__(
        self,
        whisper_model_name: str = "large-v3",
        device: Optional[str] = None,
        compute_type: Optional[str] = None,
        language: str = "en",
    ) -> None:
        self.whisper_model_name = whisper_model_name
        self.language = language
        self.device = device
        self.compute_type = compute_type
        self.asr_model = None
        self.align_model = None
        self.align_metadata = None

    def _lazy_init_whisperx(self) -> None:
        if self.asr_model is not None and self.align_model is not None:
            return
        import torch
        import whisperx

        if self.device is None:
            self.device = "cuda" if torch.cuda.is_available() else "cpu"
        if self.compute_type is None:
            self.compute_type = "float16" if self.device == "cuda" else "int8"

        print(f"Loading WhisperX ({self.whisper_model_name}) on {self.device} ({self.compute_type})...")
        self.asr_model = whisperx.load_model(
            self.whisper_model_name,
            device=self.device,
            compute_type=self.compute_type,
            language=self.language,
        )
        self.align_model, self.align_metadata = whisperx.load_align_model(
            language_code=self.language,
            device=self.device,
        )

    def transcribe_and_align(
        self,
        wav: np.ndarray,
        sr: int = EMO_TAG_SR,
        reference_text: Optional[str] = None,
    ) -> Dict[str, Any]:
        """Transcribe a 16 kHz float32 waveform with WhisperX and run forced alignment."""
        self._lazy_init_whisperx()
        import whisperx

        audio_f32 = wav.astype(np.float32)
        asr_res = self.asr_model.transcribe(audio_f32, batch_size=1, language=self.language)
        segments = asr_res.get("segments", [])
        asr_text = " ".join(seg.get("text", "").strip() for seg in segments).strip()

        # Align recognized segments (or align reference transcript if provided for boundary trimming)
        align_input = segments
        if not align_input and reference_text:
            align_input = [{"text": reference_text, "start": 0.0, "end": float(len(audio_f32) / sr)}]

        word_segments: List[Dict[str, Any]] = []
        if align_input:
            try:
                aligned = whisperx.align(
                    align_input,
                    self.align_model,
                    self.align_metadata,
                    audio_f32,
                    self.device,
                    return_char_alignments=False,
                )
                word_segments = [
                    w for w in aligned.get("word_segments", [])
                    if isinstance(w, dict) and w.get("start") is not None and w.get("end") is not None
                ]
            except Exception:
                word_segments = []

        return {
            "asr_text": asr_text,
            "asr_tokens": tokenize_emo_tag(asr_text),
            "word_segments": word_segments,
        }

    def clean_utterance(
        self,
        src_audio_path: str,
        reference_transcript: str,
        dst_audio_path: Optional[Path] = None,
    ) -> Dict[str, Any]:
        """
        Execute EMO-TAG Steps 1–5 on a single MELD utterance.
        Returns a structured dictionary with status ('retained' or rejection reason),
        cleaned waveform / path, and laughter removal diagnostics.
        """
        ref_tokens = tokenize_emo_tag(reference_transcript)
        if not ref_tokens:
            return {"retained": False, "reject_reason": "empty_reference_transcript"}

        if not src_audio_path or not Path(src_audio_path).exists():
            return {"retained": False, "reject_reason": "missing_audio_file"}

        # Step 1: Load as 16 kHz mono float32 waveform (Appendix A.2)
        wav, sr = sf.read(str(src_audio_path), dtype="float32", always_2d=True)
        wav = wav.mean(axis=1) if wav.shape[1] > 1 else wav[:, 0]
        if sr != EMO_TAG_SR:
            wav = _resample_audio(wav, sr, EMO_TAG_SR)
            sr = EMO_TAG_SR

        orig_duration = float(len(wav) / sr)
        if orig_duration < MIN_SPEECH_DURATION_SEC:
            return {"retained": False, "reject_reason": "empty_or_too_short_raw", "orig_duration": orig_duration}

        # Step 2: rVAD segmentation into speech and non-speech regions (Section 3.1)
        intervals = segment_audio_with_rvad(wav, sr=sr)
        speech_indices = [idx for idx, iv in enumerate(intervals) if iv.is_speech]
        if not speech_indices:
            return {"retained": False, "reject_reason": "empty_rvad_no_speech", "orig_duration": orig_duration}

        # Step 3: Transcribe each candidate speech region with WhisperX and check token overlap with reference
        matched_speech_indices: List[int] = []
        unrelated_speech_removed = 0
        for idx in speech_indices:
            iv = intervals[idx]
            s_sample = int(round(iv.start_sec * sr))
            e_sample = int(round(iv.end_sec * sr))
            seg_wav = wav[s_sample:e_sample]
            if len(seg_wav) < int(MIN_SPEECH_DURATION_SEC * sr):
                continue
            seg_res = self.transcribe_and_align(seg_wav, sr=sr)
            overlap = sum(1 for tok in seg_res["asr_tokens"] if tok in set(ref_tokens))
            if overlap > 0:
                matched_speech_indices.append(idx)
            else:
                unrelated_speech_removed += 1

        if not matched_speech_indices:
            return {
                "retained": False,
                "reject_reason": "unrelated_speech_no_token_overlap",
                "orig_duration": orig_duration,
                "unrelated_speech_removed": unrelated_speech_removed,
            }

        # Step 4: Inspect intervening non-speech intervals between first and last matched speech region.
        # Keep natural pauses/sighs (RMS <= -35 dBFS and Peak <= -25 dBFS); remove high-energy laughter tracks.
        first_sp = matched_speech_indices[0]
        last_sp = matched_speech_indices[-1]
        matched_set = set(matched_speech_indices)

        retained_indices: List[int] = []
        laughter_intervals_removed = 0
        laughter_seconds_removed = 0.0
        retained_pauses_count = 0

        # Leading/trailing non-speech outside [first_sp, last_sp] is dropped automatically
        for idx, iv in enumerate(intervals):
            if idx < first_sp or idx > last_sp:
                if not iv.is_speech and not is_natural_pause_or_sigh(iv):
                    laughter_intervals_removed += 1
                    laughter_seconds_removed += (iv.end_sec - iv.start_sec)
                continue

            if idx in matched_set:
                retained_indices.append(idx)
            elif not iv.is_speech:
                if is_natural_pause_or_sigh(iv):
                    retained_indices.append(idx)
                    retained_pauses_count += 1
                else:
                    laughter_intervals_removed += 1
                    laughter_seconds_removed += (iv.end_sec - iv.start_sec)
            else:
                # Unrelated speech segment inside utterance span -> remove
                pass

        # Appendix A.2: If selected regions and intervening intervals are continuous, crop a single interval;
        # otherwise concatenate the retained regions.
        is_contiguous = retained_indices == list(range(first_sp, last_sp + 1))
        if is_contiguous:
            s_sample = int(round(intervals[first_sp].start_sec * sr))
            e_sample = int(round(intervals[last_sp].end_sec * sr))
            stage4_wav = wav[s_sample:e_sample]
        else:
            pieces = [
                wav[int(round(intervals[i].start_sec * sr)) : int(round(intervals[i].end_sec * sr))]
                for i in retained_indices
            ]
            stage4_wav = np.concatenate(pieces, axis=0) if pieces else np.zeros(0, dtype=np.float32)

        if len(stage4_wav) < int(MIN_SPEECH_DURATION_SEC * sr):
            return {"retained": False, "reject_reason": "empty_after_laughter_removal", "orig_duration": orig_duration}

        # Step 5: Second-pass WhisperX forced alignment & boundary trimming (Appendix A.2 & Section 3.1)
        pass2_before = self.transcribe_and_align(stage4_wav, sr=sr, reference_text=reference_transcript)
        final_wav = stage4_wav
        word_segs = pass2_before["word_segments"]

        if word_segs:
            crop_start_sec = max(0.0, float(word_segs[0]["start"]) - BOUNDARY_MARGIN_SEC)
            crop_end_sec = min(float(len(stage4_wav) / sr), float(word_segs[-1]["end"]) + BOUNDARY_MARGIN_SEC)
            if crop_end_sec - crop_start_sec >= MIN_SPEECH_DURATION_SEC:
                candidate_crop = stage4_wav[int(round(crop_start_sec * sr)) : int(round(crop_end_sec * sr))]
                pass2_after = self.transcribe_and_align(candidate_crop, sr=sr)
                # Appendix A.2: accept the refined crop only if it preserves the same transcription
                if pass2_after["asr_tokens"] == pass2_before["asr_tokens"] and len(pass2_after["asr_tokens"]) > 0:
                    final_wav = candidate_crop
                    pass2_final = pass2_after
                else:
                    pass2_final = pass2_before
            else:
                pass2_final = pass2_before
        else:
            pass2_final = pass2_before

        # Final audibility & edit-distance gate (>= 2 consecutive insertions or >= 3 replacements -> reject)
        final_metrics = compute_interval_acoustic_metrics(final_wav, sr=sr, is_speech=True)
        if final_metrics.rms_dbfs <= -50.0 or len(final_wav) < int(MIN_SPEECH_DURATION_SEC * sr):
            return {"retained": False, "reject_reason": "inaudible_after_trim", "orig_duration": orig_duration}

        edit_diag = compute_word_edit_diagnostics(pass2_final["asr_tokens"], ref_tokens)
        if not edit_diag["passes_edit_gate"]:
            reason = (
                "consec_insertions>=2"
                if edit_diag["reject_insertions"]
                else ("replacements>=3" if edit_diag["reject_replacements"] else "zero_final_overlap")
            )
            return {
                "retained": False,
                "reject_reason": reason,
                "orig_duration": orig_duration,
                "clean_duration": float(len(final_wav) / sr),
                "asr_transcript": " ".join(pass2_final["asr_tokens"]),
                "ref_transcript": " ".join(ref_tokens),
                **edit_diag,
            }

        clean_duration = float(len(final_wav) / sr)
        out_path_str = None
        if dst_audio_path is not None:
            dst_audio_path = Path(dst_audio_path)
            dst_audio_path.parent.mkdir(parents=True, exist_ok=True)
            sf.write(str(dst_audio_path), np.clip(final_wav, -1.0, 1.0), sr, subtype="PCM_16")
            out_path_str = str(dst_audio_path.resolve())

        return {
            "retained": True,
            "reject_reason": "retained",
            "clean_audio_path": out_path_str,
            "orig_duration": orig_duration,
            "clean_duration": clean_duration,
            "laughter_intervals_removed": int(laughter_intervals_removed),
            "laughter_seconds_removed": round(float(laughter_seconds_removed), 4),
            "unrelated_speech_removed": int(unrelated_speech_removed),
            "retained_pauses_count": int(retained_pauses_count),
            "was_concatenated": bool(not is_contiguous),
            "asr_transcript": " ".join(pass2_final["asr_tokens"]),
            "ref_transcript": " ".join(ref_tokens),
            **edit_diag,
        }


In [ ]:
def run_emo_tag_cleaning_on_split(
    df: pd.DataFrame,
    split_name: str,
    cleaner: EmoTagWhisperXCleaner,
    clean_root: Path = CLEAN_DATA_DIR,
    overwrite: bool = False,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Apply the EMO-TAG 5-step cleaning pipeline across a MELD split (`train`, `val`, or `test`).
    Saves cleaned 16 kHz mono WAVs to `clean_root / split_name`, exports both `clean_{split}_set.csv`
    and `emo_tag_audit_{split}.csv`, and prints filtering statistics matching EMO-TAG Table 7(b).
    """
    split_out_dir = Path(clean_root) / split_name
    split_out_dir.mkdir(parents=True, exist_ok=True)
    clean_csv_path = Path(clean_root) / f"clean_{split_name}_set.csv"
    audit_csv_path = Path(clean_root) / f"emo_tag_audit_{split_name}.csv"

    if clean_csv_path.exists() and audit_csv_path.exists() and not overwrite:
        print(f"[{split_name}] Loading cached EMO-TAG cleaned split from {clean_csv_path}")
        return pd.read_csv(clean_csv_path), pd.read_csv(audit_csv_path)

    audit_records: List[Dict[str, Any]] = []
    retained_rows: List[Dict[str, Any]] = []

    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"EMO-TAG cleaning [{split_name}]"):
        dia_id = int(row["dialogue"])
        utt_id = int(row["utterance"])
        dst_wav = split_out_dir / f"dia{dia_id}_utt{utt_id}.wav"

        res = cleaner.clean_utterance(
            src_audio_path=str(row["audio_path"]),
            reference_transcript=str(row.get("text", "")),
            dst_audio_path=dst_wav,
        )

        audit_row = {
            "dialogue": dia_id,
            "utterance": utt_id,
            "speaker": row.get("speaker"),
            "emotion": row.get("emotion"),
            "raw_audio_path": row.get("audio_path"),
            **res,
        }
        audit_records.append(audit_row)

        if res.get("retained", False):
            clean_row = row.to_dict()
            clean_row["raw_audio_path"] = row.get("audio_path")
            clean_row["audio_path"] = res["clean_audio_path"]
            clean_row["orig_duration"] = res["orig_duration"]
            clean_row["duration"] = res["clean_duration"]
            clean_row["laughter_intervals_removed"] = res["laughter_intervals_removed"]
            clean_row["laughter_seconds_removed"] = res["laughter_seconds_removed"]
            clean_row["was_concatenated"] = res["was_concatenated"]
            retained_rows.append(clean_row)

    clean_df = pd.DataFrame(retained_rows)
    audit_df = pd.DataFrame(audit_records)

    clean_df.to_csv(clean_csv_path, index=False)
    audit_df.to_csv(audit_csv_path, index=False)

    n_orig = len(df)
    n_ret = len(clean_df)
    n_rem = n_orig - n_ret
    ret_pct = (100.0 * n_ret / n_orig) if n_orig else 0.0
    laugh_trimmed = int((audit_df.get("laughter_intervals_removed", pd.Series(dtype=int)).fillna(0) > 0).sum())

    print(f"\n=== EMO-TAG Filtering Summary [{split_name.upper()}] (cf. Table 7b) ===")
    print(f"Original : {n_orig:,}")
    print(f"Removed  : {n_rem:,}")
    print(f"Retained : {n_ret:,} ({ret_pct:.2f}%)")
    print(f"Clips with high-energy laughter intervals stripped: {laugh_trimmed:,}")
    print("\nBreakdown by status / rejection reason:")
    print(audit_df["reject_reason"].value_counts().to_string())
    print(f"Saved clean split -> {clean_csv_path}")
    print(f"Saved audit log   -> {audit_csv_path}\n")

    return clean_df, audit_df


In [ ]:
# Initialize EMO-TAG cleaner (WhisperX large-v3 + rVAD + -35/-25 dBFS laughter filter)
ensure_emo_tag_dependencies(install_whisperx=False, clone_rvad=True)
emo_tag_cleaner = EmoTagWhisperXCleaner(whisper_model_name="large-v3")

# Run on raw_train_set, raw_val_set, and raw_test_set to produce laugh-free MELD splits for eGeMAPSv02
clean_train_set, train_emo_tag_audit = run_emo_tag_cleaning_on_split(raw_train_set, "train", emo_tag_cleaner)
clean_val_set, val_emo_tag_audit = run_emo_tag_cleaning_on_split(raw_val_set, "val", emo_tag_cleaner)
clean_test_set, test_emo_tag_audit = run_emo_tag_cleaning_on_split(raw_test_set, "test", emo_tag_cleaner)


## 7. eGeMAPSv02 Acoustic Feature Extraction (`openSMILE`) & Training Set Integrity Filtering
Extracts the 5 core `ACOUSTIC_CUES` from the training and validation sets using `opensmile` (`eGeMAPSv02`, `Functionals`), then filters the training set to remove:
- Any utterance where **any** of the 5 `ACOUSTIC_CUES` features is `0.0` (or `NaN`).
- Any utterance with duration **`< 0.5s` or `> 20.0s`**.


In [ ]:
import opensmile

ACOUSTIC_CUES = {
    # 1. Altura do Pitch (Mediana em semitons - ref 27.5Hz)
    "pitch_level": "F0semitoneFrom27.5Hz_sma3nz_percentile50.0",
    
    # 2. Dinâmica de Pitch (Range interpercentil P80 - P20 em semitons)
    "pitch_variation": "F0semitoneFrom27.5Hz_sma3nz_pctlrange0-2",
    
    # 3. Energia Vocal (Mediana perceptual em Sones)
    "vocal_energy": "loudness_sma3_percentile50.0",
    
    # 4. Taxa de Fala / Ritmo (Proxy temporal do eGeMAPS)
    # Obs: Se notar zeros em clipes curtos pós-WhisperX, use "loudnessPeaksPerSec"
    "speaking_rate": "VoicedSegmentsPerSec",
    
    # 5. Qualidade / Tensão Vocal (Balanço espectral 50-1000Hz vs 1-5kHz em dB)
    "timbre": "alphaRatioV_sma3nz_amean",
}

ACOUSTIC_FEATURE_COLS = list(ACOUSTIC_CUES.values())
MIN_TRAIN_DURATION_SEC = 0.5
MAX_TRAIN_DURATION_SEC = 20.0

smile_extractor = opensmile.Smile(
    feature_set=opensmile.FeatureSet.eGeMAPSv02,
    feature_level=opensmile.FeatureLevel.Functionals,
)


def _extract_egemaps_for_row(task: Tuple[int, str]) -> Tuple[int, Dict[str, float], Optional[str]]:
    idx, audio_path = task
    if not audio_path or not Path(audio_path).exists():
        return idx, {col: float("nan") for col in ACOUSTIC_FEATURE_COLS}, f"Missing file: {audio_path}"
    try:
        feats_df = smile_extractor.process_file(str(audio_path))
        row_feats = {
            col: float(feats_df[col].iloc[0]) if col in feats_df.columns else float("nan")
            for col in ACOUSTIC_FEATURE_COLS
        }
        return idx, row_feats, None
    except Exception as exc:
        return idx, {col: float("nan") for col in ACOUSTIC_FEATURE_COLS}, str(exc)


def extract_egemaps_cues(
    df: pd.DataFrame,
    split_name: str,
    max_workers: int = MAX_WORKERS,
) -> pd.DataFrame:
    """Extract the 5 ACOUSTIC_CUES (eGeMAPSv02 Functionals) in parallel for a split DataFrame."""
    out = df.copy().reset_index(drop=True)

    # Ensure duration column is populated in seconds
    if "duration" not in out.columns or out["duration"].isna().any():
        if "time" in out.columns:
            out["duration"] = pd.to_numeric(out["time"], errors="coerce")
        else:
            out["duration"] = [
                float(sf.info(str(p)).duration) if p and Path(str(p)).exists() else float("nan")
                for p in out["audio_path"]
            ]

    tasks = [(idx, str(row["audio_path"])) for idx, row in out.iterrows()]
    extracted_rows: List[Optional[Dict[str, float]]] = [None] * len(out)
    errors = []

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(_extract_egemaps_for_row, t) for t in tasks]
        for fut in tqdm(as_completed(futures), total=len(futures), desc=f"Extracting eGeMAPSv02 [{split_name}]"):
            idx, feat_dict, err = fut.result()
            extracted_rows[idx] = feat_dict
            if err is not None:
                errors.append((idx, err))

    feats_table = pd.DataFrame(extracted_rows)
    for col in ACOUSTIC_FEATURE_COLS:
        out[col] = feats_table[col].values
    for alias, col in ACOUSTIC_CUES.items():
        out[alias] = out[col]

    if errors:
        print(f"[{split_name}] Warning: {len(errors)} extraction errors encountered.")
    return out


def filter_training_set_by_duration_and_nonzero_cues(
    df: pd.DataFrame,
    split_name: str = "train",
    min_dur: float = MIN_TRAIN_DURATION_SEC,
    max_dur: float = MAX_TRAIN_DURATION_SEC,
    delete_rejected_wavs_in_data_dir: bool = True,
) -> pd.DataFrame:
    """
    Check the training set by duration ([min_dur, max_dur]) and non-zero ACOUSTIC_CUES.
    Deletes any training utterance with duration < 0.5s or > 20.0s, or with any ACOUSTIC_CUES == 0.0 / NaN.
    """
    n_initial = len(df)
    dur_series = pd.to_numeric(df["duration"], errors="coerce")
    bad_short = dur_series < min_dur
    bad_long = dur_series > max_dur
    bad_dur = dur_series.isna() | bad_short | bad_long

    zero_or_nan_per_cue: Dict[str, int] = {}
    bad_cues = pd.Series(False, index=df.index)
    for alias, col in ACOUSTIC_CUES.items():
        s = pd.to_numeric(df[col], errors="coerce")
        is_zero_or_nan = s.isna() | (s == 0.0)
        zero_or_nan_per_cue[f"{alias} ({col})"] = int(is_zero_or_nan.sum())
        bad_cues = bad_cues | is_zero_or_nan

    reject_mask = bad_dur | bad_cues
    rejected_df = df.loc[reject_mask]
    filtered_df = df.loc[~reject_mask].reset_index(drop=True)

    # Optionally remove rejected exported .wav files inside DATA_DIR (never outside DATA_DIR)
    deleted_files = 0
    if delete_rejected_wavs_in_data_dir and "DATA_DIR" in globals():
        data_root = Path(DATA_DIR).resolve()
        for wav_str in rejected_df["audio_path"].dropna().unique():
            wav_p = Path(str(wav_str)).resolve()
            if wav_p.exists() and wav_p.is_relative_to(data_root):
                try:
                    wav_p.unlink()
                    deleted_files += 1
                except OSError:
                    pass

    print(f"\n=== Training Set eGeMAPSv02 & Duration Audit [{split_name}] ===")
    print(f"Initial training samples        : {n_initial:,}")
    print(f"Duration < {min_dur}s removed          : {int(bad_short.sum()):,}")
    print(f"Duration > {max_dur}s removed         : {int(bad_long.sum()):,}")
    print("Zero / NaN count per acoustic cue:")
    for k, cnt in zero_or_nan_per_cue.items():
        print(f"  - {k}: {cnt:,}")
    print(f"Total with >=1 zero/NaN feature : {int(bad_cues.sum()):,}")
    print(f"Total removed (union)           : {int(reject_mask.sum()):,} (deleted {deleted_files:,} WAV files in DATA_DIR)")
    print(f"Final retained training samples : {len(filtered_df):,} ({100.0 * len(filtered_df) / max(1, n_initial):.2f}%)\n")

    return filtered_df


# Select active train and val splits (prefers EMO-TAG clean splits if computed, else raw splits)
active_train_df = (
    clean_train_set
    if ("clean_train_set" in globals() and isinstance(clean_train_set, pd.DataFrame) and len(clean_train_set) > 0)
    else raw_train_set
)
active_val_df = (
    clean_val_set
    if ("clean_val_set" in globals() and isinstance(clean_val_set, pd.DataFrame) and len(clean_val_set) > 0)
    else raw_val_set
)
active_out_dir = (
    CLEAN_DATA_DIR
    if ("clean_train_set" in globals() and isinstance(clean_train_set, pd.DataFrame) and len(clean_train_set) > 0)
    else DATA_DIR
)

# 1. Extract eGeMAPSv02 features for train and val
train_egemaps_df = extract_egemaps_cues(active_train_df, "train")
val_egemaps_df = extract_egemaps_cues(active_val_df, "val")

# 2. Filter training set by duration [0.5s, 20.0s] and strictly non-zero eGeMAPSv02 features
train_egemaps_df = filter_training_set_by_duration_and_nonzero_cues(train_egemaps_df, "train")

# Update active split references and persist to CSV
if "clean_train_set" in globals() and isinstance(clean_train_set, pd.DataFrame) and len(clean_train_set) > 0:
    clean_train_set = train_egemaps_df
    clean_val_set = val_egemaps_df
else:
    raw_train_set = train_egemaps_df
    raw_val_set = val_egemaps_df

train_egemaps_csv = Path(active_out_dir) / "train_egemaps.csv"
val_egemaps_csv = Path(active_out_dir) / "val_egemaps.csv"
train_egemaps_df.to_csv(train_egemaps_csv, index=False)
val_egemaps_df.to_csv(val_egemaps_csv, index=False)
print(f"Saved train eGeMAPSv02 dataset -> {train_egemaps_csv} ({len(train_egemaps_df):,} rows)")
print(f"Saved val eGeMAPSv02 dataset   -> {val_egemaps_csv} ({len(val_egemaps_df):,} rows)")


## 8. Gender-Specific Quartile Classification (`F` vs. `M`) for Acoustic Cues
Computes `Q25` and `Q75` per acoustic cue separately for Female (`"F"`) and Male (`"M"`) speakers on `clean_train_set`:
- **`< Q25`** $\rightarrow$ `low`
- **`> Q75`** $\rightarrow$ `high`
- **`Q25 <= x <= Q75`** $\rightarrow$ `moderate`
- **`timbre`** (`alphaRatioV_sma3nz_amean`) is mapped to:
  - `low` $\rightarrow$ `"low (muffled/soft)"`
  - `moderate` $\rightarrow$ `"moderate (modal)"`
  - `high` $\rightarrow$ `"high (tense/bright)"`


In [ ]:
import json

# Ensure 'timbre' alias is mapped to alphaRatioV_sma3nz_amean
ACOUSTIC_CUES = {
    "pitch_level": "F0semitoneFrom27.5Hz_sma3nz_percentile50.0",
    "pitch_variation": "F0semitoneFrom27.5Hz_sma3nz_pctlrange0-2",
    "vocal_energy": "loudness_sma3_percentile50.0",
    "speaking_rate": "VoicedSegmentsPerSec",
    "timbre": "alphaRatioV_sma3nz_amean",
}

TIMBRE_LABEL_MAP: Dict[str, str] = {
    "low": "low (muffled/soft)",
    "moderate": "moderate (modal)",
    "high": "high (tense/bright)",
}


def compute_gender_acoustic_quartiles(
    train_df: pd.DataFrame,
    cues_map: Dict[str, str] = ACOUSTIC_CUES,
) -> Dict[str, Dict[str, Dict[str, float]]]:
    """
    Compute Q25 and Q75 for each acoustic cue on clean_train_set separately for 'F' and 'M'
    (plus '__GLOBAL__' fallback and an acoustic F0 midpoint for unresolved eval speakers).
    """
    if "gender" not in train_df.columns:
        raise ValueError("Column 'gender' ('F'/'M') not found in training DataFrame.")

    quartiles: Dict[str, Dict[str, Dict[str, float]]] = {"F": {}, "M": {}, "__GLOBAL__": {}}

    for gender_key in ("F", "M"):
        sub = train_df.loc[train_df["gender"] == gender_key]
        if len(sub) == 0:
            raise ValueError(f"No samples found for gender={gender_key!r} in training DataFrame.")
        for cue_alias, col_name in cues_map.items():
            src_col = col_name if col_name in sub.columns else cue_alias
            s = pd.to_numeric(sub[src_col], errors="coerce").dropna()
            s = s[s != 0.0]
            quartiles[gender_key][cue_alias] = {
                "q25": float(s.quantile(0.25)),
                "q50": float(s.quantile(0.50)),
                "q75": float(s.quantile(0.75)),
                "count": int(len(s)),
            }

    for cue_alias, col_name in cues_map.items():
        src_col = col_name if col_name in train_df.columns else cue_alias
        s = pd.to_numeric(train_df[src_col], errors="coerce").dropna()
        s = s[s != 0.0]
        quartiles["__GLOBAL__"][cue_alias] = {
            "q25": float(s.quantile(0.25)),
            "q50": float(s.quantile(0.50)),
            "q75": float(s.quantile(0.75)),
            "count": int(len(s)),
        }

    # Midpoint between Male and Female median F0 (Camada 3 acoustic fallback for eval speakers with gender=None)
    f0_m = quartiles["M"]["pitch_level"]["q50"]
    f0_f = quartiles["F"]["pitch_level"]["q50"]
    quartiles["__META__"] = {"f0_gender_boundary_st": float(0.5 * (f0_m + f0_f))}

    return quartiles


def classify_value_by_quartile(
    val: float,
    q25: float,
    q75: float,
    cue_alias: str,
) -> Optional[str]:
    """
    Rule:
      < Q25  -> 'low'
      > Q75  -> 'high'
      else   -> 'moderate'
    For 'timbre', maps low/moderate/high to:
      'low (muffled/soft)' | 'moderate (modal)' | 'high (tense/bright)'
    """
    if val is None or pd.isna(val) or float(val) == 0.0:
        return None
    v = float(val)
    if v < q25:
        tier = "low"
    elif v > q75:
        tier = "high"
    else:
        tier = "moderate"

    if cue_alias == "timbre":
        return TIMBRE_LABEL_MAP[tier]
    return tier


def apply_gender_quartile_classification(
    df: pd.DataFrame,
    quartiles: Dict[str, Dict[str, Dict[str, float]]],
    split_name: str,
    cues_map: Dict[str, str] = ACOUSTIC_CUES,
) -> pd.DataFrame:
    """Classify each acoustic cue into gender-relative quartile tiers on a split DataFrame."""
    out = df.copy().reset_index(drop=True)
    f0_boundary = quartiles.get("__META__", {}).get("f0_gender_boundary_st", 36.0)
    f0_col = cues_map["pitch_level"] if cues_map["pitch_level"] in out.columns else "pitch_level"

    # Resolve effective gender ('F' or 'M') per row; use F0 boundary only if gender is None (Camada 3)
    resolved_genders: List[str] = []
    for _, row in out.iterrows():
        g = row.get("gender")
        if g in ("F", "M"):
            resolved_genders.append(g)
        else:
            f0_val = row.get(f0_col, float("nan"))
            if pd.notna(f0_val) and float(f0_val) > 0.0:
                resolved_genders.append("F" if float(f0_val) >= f0_boundary else "M")
            else:
                resolved_genders.append("__GLOBAL__")
    out["effective_gender"] = resolved_genders

    for cue_alias, col_name in cues_map.items():
        src_col = col_name if col_name in out.columns else cue_alias
        out[cue_alias] = pd.to_numeric(out[src_col], errors="coerce")

        labels: List[Optional[str]] = []
        for idx, row in out.iterrows():
            g_key = out.at[idx, "effective_gender"]
            q_info = quartiles.get(g_key, quartiles["__GLOBAL__"]).get(cue_alias)
            labels.append(
                classify_value_by_quartile(
                    row[cue_alias],
                    q25=q_info["q25"],
                    q75=q_info["q75"],
                    cue_alias=cue_alias,
                )
            )
        out[f"{cue_alias}_cue"] = labels

    print(f"\n=== Gender-Quartile Acoustic Cue Distribution [{split_name}] (N={len(out):,}) ===")
    for cue_alias in cues_map:
        col_cue = f"{cue_alias}_cue"
        counts = out[col_cue].value_counts(dropna=False)
        pcts = (out[col_cue].value_counts(normalize=True, dropna=False) * 100.0).round(2)
        summary = ", ".join(f"{k}: {counts[k]:,} ({pcts[k]}%)" for k in counts.index)
        print(f"  - {cue_alias:<16}: {summary}")

    return out


# 1. Fit F and M quartiles strictly on clean_train_set (or active train_egemaps_df)
ref_train_df = (
    clean_train_set
    if ("clean_train_set" in globals() and isinstance(clean_train_set, pd.DataFrame) and len(clean_train_set) > 0)
    else train_egemaps_df
)
ref_val_df = (
    clean_val_set
    if ("clean_val_set" in globals() and isinstance(clean_val_set, pd.DataFrame) and len(clean_val_set) > 0)
    else val_egemaps_df
)

GENDER_ACOUSTIC_QUARTILES = compute_gender_acoustic_quartiles(ref_train_df, ACOUSTIC_CUES)

# Print fitted F vs M quartile cutoffs
quartile_rows = []
for g_key in ("F", "M"):
    for cue_alias, stats in GENDER_ACOUSTIC_QUARTILES[g_key].items():
        quartile_rows.append({
            "gender": g_key,
            "cue": cue_alias,
            "feature": ACOUSTIC_CUES[cue_alias],
            "Q25 (Low <)": round(stats["q25"], 4),
            "Q50 (Median)": round(stats["q50"], 4),
            "Q75 (High >)": round(stats["q75"], 4),
            "N_train": stats["count"],
        })
quartiles_summary_df = pd.DataFrame(quartile_rows)
print("=== Clean Train Set Quartile Cutoffs by Gender (F vs. M) ===")
print(quartiles_summary_df.to_string(index=False))

# 2. Apply quartile classification to clean_train_set and clean_val_set
clean_train_set = apply_gender_quartile_classification(ref_train_df, GENDER_ACOUSTIC_QUARTILES, "clean_train_set")
clean_val_set = apply_gender_quartile_classification(ref_val_df, GENDER_ACOUSTIC_QUARTILES, "clean_val_set")

# 3. Save quartile JSON and updated CSV manifests
quartiles_json_path = Path(active_out_dir) / "gender_acoustic_quartiles.json"
quartiles_json_path.write_text(json.dumps(GENDER_ACOUSTIC_QUARTILES, indent=2) + "\n", encoding="utf-8")

train_cues_csv = Path(active_out_dir) / "clean_train_set_with_cues.csv"
val_cues_csv = Path(active_out_dir) / "clean_val_set_with_cues.csv"
clean_train_set.to_csv(train_cues_csv, index=False)
clean_val_set.to_csv(val_cues_csv, index=False)

print(f"\nSaved gender quartiles JSON -> {quartiles_json_path}")
print(f"Saved classified train CSV  -> {train_cues_csv} ({len(clean_train_set):,} rows)")
print(f"Saved classified val CSV    -> {val_cues_csv} ({len(clean_val_set):,} rows)")
